# Deep Learning Model

In [ ]:
# imports
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError

### Dataset

In [ ]:
# data read
data = pd.read_csv('/content/processed_car_sales_data_cleaning.csv')
data.head()

In [ ]:
# Columns division by type

# numerical columns
numerical = data[['Engine size', 'Year of manufacture', 'Mileage', 'Price']]

# catigorical columns
catigorical = data[['Manufacturer', 'Model','Fuel type']]

# change data types into categorey
data[catigorical.columns] = data[catigorical.columns].astype('category')

# one hot encoding for categorical data
data = pd.get_dummies(data, columns=['Manufacturer', 'Model', 'Fuel type'], drop_first=True)

In [ ]:
# Set plot styles
plt.style.use('default')
sns.set_palette("deep")

In [ ]:
# Feature selection
X = data.drop("Price", axis=1)  # Input features
y = data["Price"]  # Target: Price

In [ ]:
# Features Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

### Model

In [ ]:
# Model Build
model = Sequential([
    Dense(128, input_shape=(X_train.shape[1],), activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1)  # Output layer for regression
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
# Training
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Predict
y_train_pred = model.predict(X_train).flatten()
y_test_pred = model.predict(X_test).flatten()

##### Model Evaluation

In [ ]:
# Model Evaluation
print("\nTrain R²:", r2_score(y_train, y_train_pred))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred)))
print("Test R²:", r2_score(y_test, y_test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred)))

In [ ]:
# Learning Curve
# Plot 1: Learning Curve (Loss & MAE)

plt.figure(figsize=(12, 5))

# Plot training & validation loss (MSE)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.title('Learning Curve - Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()

# Plot training & validation MAE
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Learning Curve - MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

# Plot 2: Actual vs Predicted on Test Set

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_test_pred, alpha=0.6, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title("Test Data: Actual vs Predicted Price")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Model save
model.save("DL/price_dl_model.h5")

In [ ]:
# Save metadata
features_list = data.drop("Price", axis=1).columns.tolist()
model_metadata = {
    'features': features_list,
    'targets': ['Price'],
    'model_types': ['DeepLearning (Keras Sequential)']
}

with open('DL/model_metadata.json', 'w') as f:
    json.dump(model_metadata, f)

### Prediction

In [ ]:
# Load the model
model = load_model('DL/price_dl_model.h5', compile=False)

# Recompile the model after loading
model.compile(optimizer='adam', loss=MeanSquaredError())

In [ ]:
# Create example data for prediction
feature_cols = ["Engine size", "Year of manufacture", "Mileage",
 "Manufacturer_Ford", "Manufacturer_Porsche", "Manufacturer_Toyota", "Manufacturer_VW",
 "Model_911", "Model_Cayenne", "Model_Fiesta", "Model_Focus", "Model_Golf", "Model_M5",
 "Model_Mondeo", "Model_Passat", "Model_Polo", "Model_Prius", "Model_RAV4", "Model_X3",
 "Model_Yaris", "Model_Z4",
 "Fuel type_Hybrid", "Fuel type_Petrol"]

example_data = pd.DataFrame([
    [3.0, 2000, 0, 0,0,1,0, 0,0,0,0,0,0,0,0,0,0,0,0,0,0, 0,1],   # New Toyota Corolla, Petrol
    [3.5, 2025, 0, 0,1,0,0, 1,0,0,0,0,0,0,0,0,0,0,0,0,0, 0,1],   # New Porsche 911, Petrol
    [2.5, 2025, 0, 0,0,0,1, 0,0,0,0,1,0,0,0,0,0,0,0,0,0, 1,0],   # New VW Golf, Hybrid
], columns=feature_cols)

In [ ]:
# Scaling Example Data
X_full = data.drop("Price", axis=1)
scaler = StandardScaler()
scaler.fit(X_full)  # Fit using full training data
example_scaled = scaler.transform(example_data)

In [ ]:
# Make predictions
predicted_prices = model.predict(example_scaled).flatten()

# Decode categorical columns
decoded_rows = []
for i, row in example_data.iterrows():
    manufacturer = "Unknown"
    for m in ["Ford", "Porsche", "Toyota", "VW"]:
        if row.get(f"Manufacturer_{m}", 0) == 1:
            manufacturer = m

    model_name = "Unknown"
    for m in ["911", "Cayenne", "Fiesta", "Focus", "Golf", "M5", "Mondeo",
              "Passat", "Polo", "Prius", "RAV4", "X3", "Yaris", "Z4"]:
        if row.get(f"Model_{m}", 0) == 1:
            model_name = m

    fuel = "Hybrid" if row.get("Fuel type_Hybrid", 0) == 1 else (
        "Petrol" if row.get("Fuel type_Petrol", 0) == 1 else "Other"
    )

    decoded_rows.append({
        "Engine size": row["Engine size"],
        "Year of manufacture": row["Year of manufacture"],
        "Mileage": row["Mileage"],
        "Manufacturer": manufacturer,
        "Model": model_name,
        "Fuel type": fuel,
        "Predicted Price": predicted_prices[i]
    })

# Display predictions
results_df = pd.DataFrame(decoded_rows)
print("\nPredictions with readable labels (DL):")
print(results_df)